# Análisis exploratorio de datos — Bienestar estudiantil

Este notebook organiza el trabajo en dos bloques:

1. **Comprensión general** — estructura del dataset, dimensiones, tipos de variables y primeras observaciones.
2. **Limpieza básica** — identificación y tratamiento de valores faltantes, duplicados, ruido en texto e inconsistencias en variables numéricas.

In [1]:
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
%pip install -q openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
df1 = pd.read_excel(
    r"Data\Consolidado - Información de Caracterización (2013 - 2025) (VE) (26.1).xlsx"
)
df2 = pd.read_excel(
    r"Data\Consolidado - Registros de Asistencia (2015-2022) (VE) (26.1).xlsx"
)
df3 = pd.read_excel(
    r"Data\Consolidado - Registros de Asistencias (2023 - 20251) (VE) (26.1).xlsx"
)

## 1. Comprensión general

Dimensión del problema, tipos de datos, muestra de filas, estadísticos de variables numéricas, cardinalidad en texto y observaciones iniciales **sobre los datos tal como vienen del archivo** (antes de la limpieza de la sección 2).

In [4]:
def resumen_comprension(df, nombre):
    print("\n" + "=" * 72)
    print(f" {nombre}")
    print("=" * 72)
    print(f"Dimensiones (filas × columnas): {df.shape[0]:,} × {df.shape[1]}")
    mem_mb = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"Memoria aproximada (deep): {mem_mb:.2f} MiB\n")
    print("Tipos de datos por columna:")
    print(df.dtypes.to_string())
    print("\nConteo de valores no nulos por columna:")
    print(df.count().to_string())
    print("\nPrimeras filas:")
    print(df.head(3).to_string())
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols):
        print("\nResumen estadístico (variables numéricas):")
        print(df[num_cols].describe().T.to_string())
    obj_cols = df.select_dtypes(include="object").columns
    if len(obj_cols):
        print("\nCardinalidad — valores únicos en columnas de texto (orden descendente):")
        print(df[obj_cols].nunique(dropna=True).sort_values(ascending=False).to_string())
    print("\n--- Primeras observaciones ---")
    print(
        "- Compare el tamaño de cada tabla con el proceso que representa (caracterización vs. asistencia).\n"
        "- Los tipos `object` suelen ser texto; revise si algún identificador numérico debería tratarse como categoría.\n"
        "- Muchos nulos en columnas de doble programa / secundario suelen ser esperables (no aplica a todos).\n"
        "- Los promedios académicos (PAM/PCP) deberían alinearse con la escala institucional (p. ej. 0–5)."
    )


for etiqueta, d in [
    ("Dataset 1 — Caracterización (2013–2025)", df1),
    ("Dataset 2 — Asistencia (2015–2022)", df2),
    ("Dataset 3 — Asistencia (2023–2025)", df3),
]:
    resumen_comprension(d, etiqueta)


 Dataset 1 — Caracterización (2013–2025)
Dimensiones (filas × columnas): 212,263 × 16
Memoria aproximada (deep): 132.85 MiB

Tipos de datos por columna:
Ciclo lectivo                                  object
Código                                         object
Ciudad Dirección física                        object
SEXO                                           object
Organización académica                         object
Grupo Académico                                object
Programa Académico Principal                    int64
Descripción programa  Principal                object
Promedio acumulado Programa principal(PAM)    float64
Promedio semestral Programa principal(PCP)    float64
Ubicación semestral                            object
Doble programa                                 object
Estado Doble Programa                          object
Programa Académico Secundario                 float64
Descripción programa  Secundario               object
Créditos inscritos                  

## 2. Limpieza básica

- **Valores faltantes**: conteo y porcentaje por columna; se documentan; no se imputa sin criterio de negocio (los nulos estructurales se conservan).
- **Duplicados**: filas idénticas en todas las columnas; se eliminan conservando la primera ocurrencia.
- **Ruido en texto**: espacios sobrantes, mayúsculas y tildes inconsistentes; se unifica con normalización Unicode (NFKD, sin marcas combinantes) y nombres de columna estables (`snake_case`).
- **Inconsistencias numéricas**: conteo de notas (PAM/PCP) fuera del intervalo [0, 5] cuando aplique el nombre de columna.

In [5]:
def limpiar_string(x, espacios_a_guion_bajo=False):
    if not isinstance(x, str):
        return x
    nfkd = unicodedata.normalize("NFKD", x)
    sin_acentos = "".join(c for c in nfkd if unicodedata.category(c) != "Mn")
    s = sin_acentos.lower().strip()
    if espacios_a_guion_bajo:
        s = re.sub(r"\s+", "_", s)
        s = re.sub(r"_+", "_", s).strip("_")
    return s


def limpiar_nombres_columnas(df):
    out = df.copy()
    out.columns = [limpiar_string(c, espacios_a_guion_bajo=True) for c in out.columns]
    return out


def limpiar_df(df):
    out = limpiar_nombres_columnas(df)
    for col in out.select_dtypes(include="object").columns:
        out[col] = out[col].map(limpiar_string)
    return out


def tabla_faltantes(df):
    n = len(df)
    t = df.isnull().sum()
    p = (t / n * 100).round(2)
    tab = pd.DataFrame({"nulos": t, "pct": p})
    return tab[tab["nulos"] > 0].sort_values("nulos", ascending=False)


def conteo_duplicados_fila_completa(df):
    return int(df.duplicated().sum())


def celdas_texto_con_espacios_extremos(df):
    total = 0
    for col in df.select_dtypes(include="object").columns:
        s = df[col].dropna()
        total += int(s.map(lambda x: isinstance(x, str) and x != x.strip()).sum())
    return total


def conteo_notas_fuera_rango(df, cmin=0.0, cmax=5.0):
    filas = {}
    for col in df.select_dtypes(include=[np.number]).columns:
        cl = col.lower()
        if "pam" in cl or "pcp" in cl:
            mask = df[col].notna() & ((df[col] < cmin) | (df[col] > cmax))
            n = int(mask.sum())
            if n:
                filas[col] = n
    return filas

In [6]:
print("Diagnóstico previo a normalizar (datos crudos)\n")

for nombre, d in [
    ("df1 — Caracterización", df1),
    ("df2 — Asistencia 2015–2022", df2),
    ("df3 — Asistencia 2023–2025", df3),
]:
    print("-" * 72)
    print(nombre)
    print("Faltantes (columnas con al menos un nulo):")
    tf = tabla_faltantes(d)
    print(tf.to_string() if len(tf) else "  (ninguno)")
    dup = conteo_duplicados_fila_completa(d)
    print(f"Filas duplicadas (todas las columnas): {dup:,}")
    esp = celdas_texto_con_espacios_extremos(d)
    print(f"Celdas de texto con espacios al inicio/fin: {esp:,}")
    out = conteo_notas_fuera_rango(d)
    print(
        "Notas numéricas fuera de [0, 5] (columnas PAM/PCP):",
        out if out else "ninguna detectada",
    )
    print()

Diagnóstico previo a normalizar (datos crudos)

------------------------------------------------------------------------
df1 — Caracterización
Faltantes (columnas con al menos un nulo):
                                             nulos    pct
Programa Académico Secundario               203105  95.69
Descripción programa  Secundario            203105  95.69
Estado Doble Programa                       203016  95.64
Promedio semestral Programa principal(PCP)   33442  15.75
Ciudad Dirección física                      13216   6.23
Ubicación semestral                           8439   3.98
Promedio acumulado Programa principal(PAM)    4968   2.34
Organización académica                           9   0.00
Filas duplicadas (todas las columnas): 0
Celdas de texto con espacios al inicio/fin: 0
Notas numéricas fuera de [0, 5] (columnas PAM/PCP): ninguna detectada

------------------------------------------------------------------------
df2 — Asistencia 2015–2022
Faltantes (columnas con al menos u

In [ ]:
df1 = limpiar_df(df1)
df2 = limpiar_df(df2)
df3 = limpiar_df(df3)

for d, nombre in [(df1, "df1"), (df2, "df2"), (df3, "df3")]:
    antes = len(d)
    n_dup = d.duplicated().sum()
    d.drop_duplicates(inplace=True)
    despues = len(d)
    print(
        f"{nombre}: filas antes {antes:,} | duplicadas eliminadas {int(n_dup):,} | filas después {despues:,}"
    )

print("\nTras limpieza — muestra df1 (primeras filas):")
print(df1.head(3).to_string())

df1: filas antes 212,263 | duplicadas eliminadas 0 | filas después 212,263
df2: filas antes 220,339 | duplicadas eliminadas 0 | filas después 220,339
df3: filas antes 87,127 | duplicadas eliminadas 0 | filas después 87,127

Tras limpieza — muestra df1 (primeras filas):
    ciclo_lectivo            codigo ciudad_direccion_fisica   sexo        organizacion_academica               grupo_academico  programa_academico_principal  descripcion_programa_principal  promedio_acumulado_programa_principal(pam)  promedio_semestral_programa_principal(pcp) ubicacion_semestral doble_programa estado_doble_programa  programa_academico_secundario descripcion_programa_secundario  creditos_inscritos
0  periodo 2015-1  c101489ho303414h             bogota d.c.  mujer                    psicologia                    psicologia                             8                      psicologia                                        4.33                                        4.29          semestre 7             si  

In [8]:
tf1 = tabla_faltantes(df1)
print("Faltantes después de limpiar (df1):")
print(tf1.to_string() if len(tf1) else "(ninguno)")
print("\nInconsistencias PAM/PCP tras limpiar (df1):")
print(conteo_notas_fuera_rango(df1) or "ninguna detectada")

Faltantes después de limpiar (df1):
                                             nulos    pct
programa_academico_secundario               203105  95.69
descripcion_programa_secundario             203105  95.69
estado_doble_programa                       203016  95.64
promedio_semestral_programa_principal(pcp)   33442  15.75
ciudad_direccion_fisica                      13216   6.23
ubicacion_semestral                           8439   3.98
promedio_acumulado_programa_principal(pam)    4968   2.34
organizacion_academica                           9   0.00

Inconsistencias PAM/PCP tras limpiar (df1):
ninguna detectada


## 3. Integración de `df1` y `df2` en un solo DataFrame

Objetivo: crear un `df12` completo sin columnas duplicadas.

Estrategia aplicada:
- Usar `codigo` como llave de cruce.
- Reducir `df2` a una fila por `codigo` (evita multiplicar filas al hacer `merge`).
- Hacer `left merge` para conservar el universo de `df1`.
- Resolver columnas comunes con prioridad en `df1` y completar faltantes con `df2`.
- Eliminar columnas auxiliares con sufijos (`_df1`, `_df2`) para dejar una sola versión por campo.

In [9]:
# -------- Integración df1 + df2 sin duplicar columnas --------

# 1) df2 puede tener varias filas por estudiante (codigo) por diferentes actividades.
#    Para una integración de "perfil" en una sola fila por estudiante, nos quedamos
#    con el primer registro no nulo por columna dentro de cada codigo.
def primer_no_nulo(serie):
    s = serie.dropna()
    return s.iloc[0] if not s.empty else np.nan

# Agrupamos df2 para asegurar unicidad de la llave de merge.
df2_unico = (
    df2.groupby("codigo", as_index=False)
    .agg({col: primer_no_nulo for col in df2.columns if col != "codigo"})
)

# 2) Merge conservando todas las filas de df1 (base principal de caracterización).
#    Usamos sufijos para identificar columnas comunes temporalmente.
df12_tmp = df1.merge(df2_unico, on="codigo", how="left", suffixes=("_df1", "_df2"))

# 3) Resolver columnas repetidas:
#    - Si una columna existe en ambos dataframes, se crea una sola columna final.
#    - Prioridad: valor de df1; si está vacío, usar df2.
columnas_comunes = [c for c in df1.columns if c in df2_unico.columns and c != "codigo"]

for c in columnas_comunes:
    c1 = f"{c}_df1"
    c2 = f"{c}_df2"
    if c1 in df12_tmp.columns and c2 in df12_tmp.columns:
        df12_tmp[c] = df12_tmp[c1].combine_first(df12_tmp[c2])

# 4) Quitar columnas auxiliares con sufijos y dejar dataset final limpio.
cols_aux = [c for c in df12_tmp.columns if c.endswith("_df1") or c.endswith("_df2")]
df12 = df12_tmp.drop(columns=cols_aux)

# 5) Verificaciones rápidas.
print(f"Filas df1: {len(df1):,}")
print(f"Filas df2 (limpio): {len(df2):,}")
print(f"Filas df2_unico (por codigo): {len(df2_unico):,}")
print(f"Filas df12 final: {len(df12):,}")
print(f"Columnas df12 final: {df12.shape[1]}")

# Validar que no queden nombres de columnas repetidos.
print("¿Hay columnas duplicadas en df12?:", df12.columns.duplicated().any())

# Vista rápida del resultado integrado.
print("\nMuestra df12:")
print(df12.head(3).to_string())

Filas df1: 212,263
Filas df2 (limpio): 220,339
Filas df2_unico (por codigo): 41,163
Filas df12 final: 212,263
Columnas df12 final: 21
¿Hay columnas duplicadas en df12?: False

Muestra df12:
    ciclo_lectivo            codigo ciudad_direccion_fisica   sexo        organizacion_academica               grupo_academico  programa_academico_principal  descripcion_programa_principal  promedio_acumulado_programa_principal(pam)  promedio_semestral_programa_principal(pcp) ubicacion_semestral doble_programa estado_doble_programa  programa_academico_secundario descripcion_programa_secundario  creditos_inscritos periodo     ano                    jefatura               estrategia_programa_o_servicio                                 actividad
0  periodo 2015-1  c101489ho303414h             bogota d.c.  mujer                    psicologia                    psicologia                             8                      psicologia                                        4.33                              

## 4. Análisis estadístico descriptivo

Objetivo: resumir el comportamiento de las variables de `df12` con medidas de tendencia central, dispersión y forma de distribución según tipo de variable.

- **Numéricas**: media, mediana, moda, desviación estándar, IQR, asimetría y curtosis.
- **Categóricas**: frecuencia absoluta, porcentaje y número de categorías únicas.

In [11]:
# -------- Estadística descriptiva de df12 --------

# Separación por tipo de variable
num_cols = df12.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df12.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"Variables numéricas: {len(num_cols)}")
print(f"Variables categóricas/texto: {len(cat_cols)}")

# Resumen numérico avanzado
if num_cols:
    desc_num = df12[num_cols].describe().T
    moda_num = df12[num_cols].mode(dropna=True).iloc[0] if not df12[num_cols].mode(dropna=True).empty else pd.Series(index=num_cols, dtype=float)

    resumen_num = pd.DataFrame({
        "count": desc_num["count"],
        "mean": desc_num["mean"],
        "median": df12[num_cols].median(numeric_only=True),
        "mode": moda_num.reindex(num_cols),
        "std": desc_num["std"],
        "min": desc_num["min"],
        "q1": desc_num["25%"],
        "q3": desc_num["75%"],
        "max": desc_num["max"],
        "iqr": desc_num["75%"] - desc_num["25%"],
        "skew": df12[num_cols].skew(numeric_only=True),
        "kurtosis": df12[num_cols].kurtosis(numeric_only=True),
    }).sort_index()

    print("\nResumen descriptivo (numéricas):")
    print(resumen_num.round(4).to_string())
else:
    resumen_num = pd.DataFrame()
    print("No hay variables numéricas en df12.")

# Resumen categórico
if cat_cols:
    resumen_cat = pd.DataFrame({
        "n_no_nulos": [df12[c].notna().sum() for c in cat_cols],
        "n_unicos": [df12[c].nunique(dropna=True) for c in cat_cols],
        "moda": [df12[c].mode(dropna=True).iloc[0] if not df12[c].mode(dropna=True).empty else np.nan for c in cat_cols],
        "freq_moda": [df12[c].value_counts(dropna=True).iloc[0] if df12[c].notna().any() else 0 for c in cat_cols],
    }, index=cat_cols)

    resumen_cat["pct_moda"] = (resumen_cat["freq_moda"] / len(df12) * 100).round(2)

    print("\nResumen descriptivo (categóricas):")
    print(resumen_cat.sort_values("n_unicos", ascending=False).to_string())
else:
    resumen_cat = pd.DataFrame()
    print("No hay variables categóricas en df12.")

Variables numéricas: 6
Variables categóricas/texto: 15

Resumen descriptivo (numéricas):
                                               count       mean   median    mode       std      min       q1       q3     max     iqr    skew  kurtosis
ano                                         170356.0  2018.3521  2018.00  2017.0    1.8008  2015.00  2017.00  2020.00  2022.0    3.00  0.5618   -0.6839
creditos_inscritos                          212263.0    16.1736    18.00    18.0    7.5086     0.00    15.00    20.00    43.0    5.00 -0.8922    0.7459
programa_academico_principal                212263.0   145.7539   146.00    46.0  170.4083     1.00     8.00   151.00   814.0  143.00  1.6296    1.7066
programa_academico_secundario                 9158.0   245.2728   151.00   541.0  223.6250     5.00    46.00   457.00   812.0  411.00  0.6548   -0.8375
promedio_acumulado_programa_principal(pam)  207295.0     3.9342     3.97     4.1    0.4104     0.02     3.72     4.20     5.0    0.48 -1.8359   10.5957

## 5. Visualizaciones interpretativas

Se generan gráficos para comunicar hallazgos y se guardan automáticamente en `src/graphics`.

Gráficos incluidos:
- Histogramas de variables numéricas.
- Boxplots de variables numéricas (detección visual de outliers).
- Scatter plot entre dos variables numéricas relevantes (si existen).
- Barras para categorías más frecuentes en variables categóricas.

In [12]:
# -------- Visualizaciones y exportación a src/graphics --------
from pathlib import Path

out_dir = Path("src") / "graphics"
out_dir.mkdir(parents=True, exist_ok=True)

saved_files = []

# 1) Histogramas para variables numéricas (máximo 6 para mantener legibilidad)
num_plot_cols = num_cols[:6]
for col in num_plot_cols:
    serie = df12[col].dropna()
    if serie.empty:
        continue
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(serie, bins=30, edgecolor="black")
    ax.set_title(f"Histograma - {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Frecuencia")
    fig.tight_layout()
    file_path = out_dir / f"hist_{col}.png"
    fig.savefig(file_path, dpi=150)
    plt.close(fig)
    saved_files.append(str(file_path))

# 2) Boxplots para variables numéricas (máximo 6)
for col in num_plot_cols:
    serie = df12[col].dropna()
    if serie.empty:
        continue
    fig, ax = plt.subplots(figsize=(8, 2.8))
    ax.boxplot(serie, vert=False)
    ax.set_title(f"Boxplot - {col}")
    ax.set_xlabel(col)
    fig.tight_layout()
    file_path = out_dir / f"box_{col}.png"
    fig.savefig(file_path, dpi=150)
    plt.close(fig)
    saved_files.append(str(file_path))

# 3) Scatter plot entre dos variables numéricas de interés (si existen)
x_candidate = "promedio_acumulado_programa_principal(pam)"
y_candidate = "promedio_semestral_programa_principal(pcp)"

if x_candidate in df12.columns and y_candidate in df12.columns:
    temp = df12[[x_candidate, y_candidate]].dropna()
    if not temp.empty:
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(temp[x_candidate], temp[y_candidate], alpha=0.25, s=10)
        ax.set_title("Relación PAM vs PCP")
        ax.set_xlabel(x_candidate)
        ax.set_ylabel(y_candidate)
        fig.tight_layout()
        file_path = out_dir / "scatter_pam_vs_pcp.png"
        fig.savefig(file_path, dpi=150)
        plt.close(fig)
        saved_files.append(str(file_path))

# 4) Barras de categorías más frecuentes (top 10) para variables categóricas (máximo 4)
cat_plot_cols = cat_cols[:4]
for col in cat_plot_cols:
    freq = df12[col].value_counts(dropna=True).head(10)
    if freq.empty:
        continue
    fig, ax = plt.subplots(figsize=(9, 4.5))
    freq.sort_values().plot(kind="barh", ax=ax)
    ax.set_title(f"Top 10 categorías - {col}")
    ax.set_xlabel("Frecuencia")
    ax.set_ylabel(col)
    fig.tight_layout()
    file_path = out_dir / f"bar_top10_{col}.png"
    fig.savefig(file_path, dpi=150)
    plt.close(fig)
    saved_files.append(str(file_path))

print("Gráficos guardados en:", out_dir)
print(f"Total de imágenes generadas: {len(saved_files)}")
for p in saved_files:
    print("-", p)

Gráficos guardados en: src\graphics
Total de imágenes generadas: 17
- src\graphics\hist_programa_academico_principal.png
- src\graphics\hist_promedio_acumulado_programa_principal(pam).png
- src\graphics\hist_promedio_semestral_programa_principal(pcp).png
- src\graphics\hist_programa_academico_secundario.png
- src\graphics\hist_creditos_inscritos.png
- src\graphics\hist_ano.png
- src\graphics\box_programa_academico_principal.png
- src\graphics\box_promedio_acumulado_programa_principal(pam).png
- src\graphics\box_promedio_semestral_programa_principal(pcp).png
- src\graphics\box_programa_academico_secundario.png
- src\graphics\box_creditos_inscritos.png
- src\graphics\box_ano.png
- src\graphics\scatter_pam_vs_pcp.png
- src\graphics\bar_top10_ciclo_lectivo.png
- src\graphics\bar_top10_codigo.png
- src\graphics\bar_top10_ciudad_direccion_fisica.png
- src\graphics\bar_top10_sexo.png
